In [0]:
import pyspark.sql.functions as f
import pyspark.sql.types as t

In [0]:
%sql
SELECT 
    raceId,
    round,
    circuitId,
    CAST(date_time AS DATE) AS date,
    name 
FROM f1_racing.silver.slv_races

In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW winner_view AS

SELECT 
    race.raceId,
    race.circuitId,
    race.round,
    cast(race.date_time as date),
    race.name,
    res.driver_id as winner_id
FROM f1_racing.silver.slv_races AS race
LEFT JOIN f1_racing.silver.slv_results AS res
ON race.raceId = res.race_id
WHERE res.position = 1 

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW runner_view AS

SELECT 
    race.raceId,
    race.circuitId,
    race.round,
    cast(race.date_time as date),
    race.name,
    res.driver_id as runner_id
FROM f1_racing.silver.slv_races AS race
LEFT JOIN f1_racing.silver.slv_results AS res
ON race.raceId = res.race_id
WHERE res.position = 2

In [0]:
df = spark.sql("""
            select  
                win.*,
                run.runner_id
            from winner_view as win 
            left join runner_view as run
            on win.raceId = run.raceId
        """)

In [0]:
(df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("f1_racing.gold.gold_races"))